In [1]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import KFold
import sklearn.linear_model as lm

from utils import Preprocessor, compute_error

seed = 42
np.random.seed(seed)

sns.set_style('darkgrid')
sns.set_theme(font_scale=1.5)


df = pd.read_csv("data/heartDisease.csv")
df = df.drop(labels=["row.names"], axis=1)


In [5]:

best_lm = lm.LogisticRegression(penalty="l2", C=1/10)

preprocessor = Preprocessor(task='classification')

X_preprocessed, y = preprocessor.fit_transform(df)

best_lm.fit(X_preprocessed, y)


for coef, feature_name in zip(best_lm.coef_.reshape(-1), preprocessor.get_feature_names_out().tolist()):
    print(feature_name, "\t\t", coef)





num__sbp 		 0.15373836964672918
num__typea 		 0.30746490362344614
num__age 		 0.6355489305608789
num__obesity 		 -0.1089516622107128
num__alcohol 		 -0.044145063143873474
num__ldl 		 0.31086972654510653
num__tobacco 		 0.3690795771532169
cat__famhist 		 0.6058602973107807


/Users/krusand/Documents/GitHub/02452-ML-Project/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [20]:
coef_sbp = 0.15373836964672918
coef_typea = 0.30746490362344614
coef_age = 0.6355489305608789
coef_obesity = 0.1089516622107128
coef_alcohol = 0.044145063143873474
coef_ldl = 0.31086972654510653
coef_tobacco = 0.3690795771532169
coef_famhist = 0.6058602973107807

In [36]:
row_sample = X_preprocessed.iloc[1:2]
row_sample

,num__sbp,num__typea,num__age,num__obesity,num__alcohol,num__ldl,num__tobacco,cat__famhist
1,0.277089,0.193344,1.383115,0.671373,0.273101,-0.15968,-0.491199,0.0


In [43]:
best_lm.intercept_

array([-1.09691429])

In [42]:
pred_val = (best_lm.intercept_
 + coef_sbp * row_sample["num__sbp"].values 
 + coef_typea * row_sample["num__typea"].values 
 + coef_age * row_sample["num__age"].values 
 + coef_obesity * row_sample["num__obesity"].values 
 + coef_alcohol * row_sample["num__alcohol"].values 
 + coef_ldl * row_sample["num__ldl"].values 
 + coef_tobacco * row_sample["num__tobacco"].values 
 + coef_famhist * row_sample["cat__famhist"].values 
)

def sigmoid(z):
    return 1/(1 + np.exp(-z))

sigmoid(pred_val)

array([0.43498035])

In [63]:
print(r"x_{i, \text{sbp}}", round(row_sample["num__sbp"].values[0], 2))
print(r"x_{i, \text{typea}}", round(row_sample["num__typea"].values[0], 2))
print(r"x_{i, \text{age}}", round(row_sample["num__age"].values[0], 2))
print(r"x_{i, \text{obesity}}", round(row_sample["num__obesity"].values[0], 2))
print(r"x_{i, \text{alcohol}}", round(row_sample["num__alcohol"].values[0], 2))
print(r"x_{i, \text{ldl}}", round(row_sample["num__ldl"].values[0], 2))
print(r"x_{i, \text{tobacco}}", round(row_sample["num__tobacco"].values[0], 2))
print(r"x_{i, \text{famhist}}", round(row_sample["cat__famhist"].values[0], 2))

x_{i, \text{sbp}} 0.28
x_{i, \text{typea}} 0.19
x_{i, \text{age}} 1.38
x_{i, \text{obesity}} 0.67
x_{i, \text{alcohol}} 0.27
x_{i, \text{ldl}} -0.16
x_{i, \text{tobacco}} -0.49
x_{i, \text{famhist}} 0.0


In [50]:
pred_val

array([-0.26155967])

$$
\begin{align*}
w_\text{intercept} = -1.10 &\\
w_{\text{sbp}} = 0.15 &\qquad x_{i, \text{sbp}} =  0.28  \\
w_{\text{typea}} = 0.31 &\qquad x_{i, \text{typea}} =  0.19 \\
w_{\text{age}} = 0.64 &\qquad x_{i, \text{age}} =  1.38 \\
w_{\text{obesity}} = 0.11 &\qquad x_{i, \text{obesity}} =  0.67 \\
w_{\text{alcohol}} = 0.04 &\qquad x_{i, \text{alcohol}} =  0.27 \\
w_{\text{ldl}} = 0.31 &\qquad x_{i, \text{ldl}} = -0.16 \\
w_{\text{tobacco}} = 0.37 &\qquad x_{i, \text{tobacco}} = -0.49 \\
w_{\text{famhist}} = 0.61 &\qquad x_{i, \text{famhist} = } 0.0
\end{align*}
$$

$$
\begin{align*}
y_i &= \sigma (w_{\text{intercept}} 
+ w_{\text{sbp}} \cdot x_{i, \text{sbp}}
+ w_{\text{typea}} \cdot x_{i, \text{typea}}
+ w_{\text{age}} \cdot x_{i, \text{age}} \\
&\quad + w_{\text{obesity}} \cdot x_{i, \text{obesity}}
+ w_{\text{alcohol}} \cdot x_{i, \text{alcohol}}
+ w_{\text{ldl}} \cdot x_{i, \text{ldl}} \\
&\quad + w_{\text{tobacco}} \cdot x_{i, \text{tobacco}}
+ w_{\text{famhist}} \cdot x_{i, \text{famhist}}) \\
&= \sigma(-1.10
+ 0.15 \cdot 0.28
+ 0.31 \cdot 0.19
+ 0.64 \cdot 1.38 \\
&\quad + 0.11 \cdot 0.67
+ 0.04 \cdot 0.27
+ 0.31 \cdot -0.16 \\
&\quad+ 0.37 \cdot -0.49
+ 0.61 \cdot 0.0) \\
&= \frac{1}{1+\exp^{-(-0.26)}} \\
&= 0.44
\end{align*} 
$$ 